# Stage 0b: Zoom In's "superposition" car circuit

Zoom In (2020) showed a car detector in `mixed4c` whose feature is spread into dog neurons in
`mixed4d`, read as the car being stored in superposition. The Goodfire essay says those are
generalized pose neurons and nothing reads the combination back out.

Selection rules and predictions are fixed in `prereg/zoomin_car.json` and `PREREGISTRATION.md`
(committed before this notebook was run). Runtime: **A100**. Reuses the Stage 0 cache on Drive.

In [ ]:
SMOKE = False
REPO_URL = "https://github.com/TheoHLong/already-clean-check.git"
DRIVE_CODE = "/content/drive/MyDrive/already-clean-check"
RUN_ROOT = "/content/drive/MyDrive/acc_runs/stage0"
CACHE_A = RUN_ROOT + "/cache_inceptionv1"        # mixed4d, mixed4e (from Stage 0)
CACHE_B = RUN_ROOT + "/cache_inceptionv1_4c"     # mixed4c (new)
OUT = RUN_ROOT + "/zoomin_car"
BATCH = 256

In [ ]:
import os, sys, subprocess, json
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CODE = DRIVE_CODE if os.path.isdir(DRIVE_CODE) else "/content/already-clean-check"
    if not os.path.isdir(CODE):
        subprocess.run(["git", "clone", "-q", REPO_URL, CODE], check=True)
    else:
        subprocess.run(["git", "-C", CODE, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch-lucent==0.1.8", "datasets", "huggingface_hub"], check=True)
else:
    CODE = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if SMOKE:
    RUN_ROOT = os.path.join("/content" if IN_COLAB else CODE, "smoke_run")
    CACHE_A, CACHE_B, OUT = RUN_ROOT + "/cache_a", RUN_ROOT + "/cache_b", RUN_ROOT + "/zoomin_car"
sys.path.insert(0, CODE)
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("code:", CODE, "| commit:", subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %ci"], capture_output=True, text=True).stdout.strip(), "| device:", DEVICE)

In [ ]:
from acc.data import download_hf_val, open_val, make_fake_imagefolder
if SMOKE:
    VAL_DIR = make_fake_imagefolder(os.path.join(RUN_ROOT, "fake_val"))
else:
    if IN_COLAB:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    VAL_DIR = download_hf_val("/content/imagenet_val")
source = open_val(VAL_DIR)
print(type(source).__name__, len(source), "images")

## Caches
mixed4c is new; mixed4d and mixed4e are reused from Stage 0 (resumable, on Drive).

In [ ]:
from acc.models import load_inceptionv1
from acc.cache import cache_activations, load_cache
model = load_inceptionv1(DEVICE)
W = os.cpu_count()
cache_activations(model, source, ["mixed4c"], CACHE_B, batch_size=BATCH, num_workers=W, device=DEVICE, shard_size=16 if SMOKE else 5000)
cache_activations(model, source, ["mixed4d", "mixed4e"], CACHE_A, batch_size=BATCH, num_workers=W, device=DEVICE, shard_size=16 if SMOKE else 5000)
c4c, c4d, c4e = load_cache(CACHE_B, "mixed4c"), load_cache(CACHE_A, "mixed4d"), load_cache(CACHE_A, "mixed4e")
print({k: v["max"].shape for k, v in [("4c", c4c), ("4d", c4d), ("4e", c4e)]})

## Preregistered checks (P1 to P5)
Everything printed as plain text.

In [ ]:
from acc import zoomin as Z
summary = Z.run_all(model, source, c4c, c4d, c4e, OUT, device=DEVICE, workers=W)
print("RESULTS_START")
print(json.dumps(summary, indent=1, default=float))
for f in ["car_detectors_4c", "targets_of_C_4d", "readout_4e", "combiners_4e"]:
    print("\n==", f)
    try:
        print(pd.read_csv(os.path.join(OUT, f + ".csv")).head(12).round(3).to_string())
    except Exception as e:
        print("(empty)", e)
try:
    fp = pd.read_csv(os.path.join(OUT, "flip_preference_4d.csv"))
    print("\n== flip preference of T channels"); print(fp[fp.channel.isin([int(t.split(':')[1]) for t in summary['T']])].round(2).to_string())
except Exception as e:
    print("no flip table:", e)
print("RESULTS_END")

## Look: C, the D neurons on all images and on cars only, and the combiners

In [ ]:
from acc import analysis as A
C = int(summary["C"].split(":")[1]); D = [int(d.split(":")[1]) for d in summary["D"]]
is_car4d, _ = Z.masks(c4d["label"])
A.show_examples(source, c4c, C, k=12, crop=112, title=f"car detector mixed4c:{C}"); plt.show()
cars_only = Z.subset(c4d, is_car4d)
for d in D:
    A.show_examples(source, c4d, d, k=12, crop=112, title=f"mixed4d:{d} all images"); plt.show()
    A.show_examples(source, cars_only, d, k=12, crop=None, title=f"mixed4d:{d} car images only"); plt.show()
for ch in pd.read_csv(os.path.join(OUT, "combiners_4e.csv"))["channel_4e"].head(3):
    A.show_examples(source, c4e, int(ch), k=12, crop=112, title=f"combiner mixed4e:{ch}"); plt.show()

## Feature visualisation of C and the D neurons (standard settings)

In [ ]:
from acc.stage0 import feature_vis
steps, size = (8, 64) if SMOKE else (512, 224)
units = [f"mixed4c:{C}"] + [f"mixed4d:{d}" for d in D]
fig, axes = plt.subplots(1, len(units), figsize=(3 * len(units), 3.2))
for ax, u in zip(np.atleast_1d(axes), units):
    ax.imshow(feature_vis(u, steps=steps, size=size)); ax.set_title(u); ax.axis("off")
plt.show()